# Module 8: SAT Solving and Boolean Reasoning

**ACL2 for Computer Science — University-Level Tutorial Series**

## Learning Objectives

By the end of this module you will be able to:

1. Represent boolean formulas in conjunctive normal form (CNF)
2. Evaluate CNF formulas against variable assignments
3. Implement truth-table–based tautology checking
4. Understand and implement unit propagation
5. Build a simple DPLL SAT solver in ACL2 and prove its soundness
6. Know the ACL2 community books related to SAT and BDDs

## 8.1 Introduction

**Boolean satisfiability (SAT)** is the canonical NP-complete problem: given a boolean formula, is there an assignment of truth values to its variables that makes the formula true?

SAT solving is central to:
- **Hardware verification** (checking circuit equivalence)
- **Software verification** (bounded model checking)
- **AI planning** and **scheduling**
- **Cryptanalysis**

ACL2 has extensive SAT-related books and tools, including verified SAT proof checkers that can validate proofs produced by industrial SAT solvers.

### The Landscape of SAT in ACL2

| Tool / Book | Purpose |
|---|---|
| `books/clause-processors/` | Integration of external SAT solvers |
| `books/projects/sat/` | SAT solver implementations and proofs |
| `books/centaur/satlink/` | Connecting ACL2 to external SAT solvers |
| `books/projects/sat/lrat/` | Verified LRAT proof checker |
| ACL2's built-in BDDs | Binary Decision Diagram support |

## 8.2 Boolean Formulas in CNF

We represent formulas in **Conjunctive Normal Form (CNF)** — the standard input format for SAT solvers.

**Conventions:**
- A **literal** is a non-zero integer: positive means the variable is true, negative means it is negated. E.g., `3` means $x_3$ and `-3` means $\neg x_3$.
- A **clause** is a list of literals representing their disjunction (OR).
- A **formula** is a list of clauses representing their conjunction (AND).

For example, the formula $(x_1 \vee \neg x_2) \wedge (x_2 \vee x_3)$ is represented as `((1 -2) (2 3))`.

### Variable Assignments

An **assignment** is an association list mapping variable numbers (positive integers) to booleans:

In [ ]:
; Look up a variable's value in an assignment.
; Unassigned variables default to nil.
(defun lookup-var (var assignment)
  (cond ((endp assignment) nil)
        ((equal (car (car assignment)) var)
         (cdr (car assignment)))
        (t (lookup-var var (cdr assignment)))))

In [ ]:
; Test assignment: x1=t, x2=nil, x3=t
(defconst *test-assignment*
  '((1 . t) (2 . nil) (3 . t)))

In [ ]:
; Look up some values
(list (lookup-var 1 *test-assignment*)   ; t
      (lookup-var 2 *test-assignment*)   ; nil
      (lookup-var 3 *test-assignment*)   ; t
      (lookup-var 4 *test-assignment*))  ; nil (default)

### Evaluating Formulas

In [ ]:
; Evaluate a literal under an assignment.
; A positive literal n is true iff variable n is true.
; A negative literal -n is true iff variable n is false.
(defun eval-literal (lit assignment)
  (if (< lit 0)
      (not (lookup-var (- lit) assignment))
    (lookup-var lit assignment)))

In [ ]:
; Evaluate a clause (disjunction of literals).
; A clause is true iff at least one literal is true.
(defun eval-clause (clause assignment)
  (if (endp clause)
      nil
    (if (eval-literal (car clause) assignment)
        t
      (eval-clause (cdr clause) assignment))))

In [ ]:
; Evaluate a formula (conjunction of clauses).
; A formula is true iff every clause is true.
(defun eval-formula (formula assignment)
  (if (endp formula)
      t
    (if (eval-clause (car formula) assignment)
        (eval-formula (cdr formula) assignment)
      nil)))

In [ ]:
; Test: (x1 OR NOT x2) AND (x2 OR x3)
; with x1=t, x2=nil, x3=t
(eval-formula '((1 -2) (2 3)) *test-assignment*)
; clause 1: x1=t so true; clause 2: x2=nil but x3=t so true → t

In [ ]:
; Test: an unsatisfied formula under this assignment
; (NOT x1) AND (x1)  → always unsatisfiable
(eval-formula '((-1) (1)) *test-assignment*)
; clause 1: NOT x1 = nil → formula is nil

## 8.3 Truth Tables and Tautology Checking

For formulas with a small number of variables, we can check all possible assignments. This gives us a **truth-table** approach to tautology checking.

A **tautology** is a formula that is true under *every* assignment.

In [ ]:
; Generate all possible assignments for variables 1..n.
; Returns a list of assignments.
(defun all-assignments-helper (n base)
  (if (zp n)
      (list base)
    (append (all-assignments-helper (- n 1)
                                     (cons (cons n nil) base))
            (all-assignments-helper (- n 1)
                                     (cons (cons n t) base)))))

In [ ]:
; Top-level: generate all 2^n assignments for n variables
(defun all-assignments (n)
  (all-assignments-helper n nil))

In [ ]:
; Test: all assignments for 2 variables
(all-assignments 2)

In [ ]:
; Check if a formula is a tautology:
; true under every assignment for variables 1..n
(defun tautologyp (formula n)
  (tautologyp-helper formula (all-assignments n)))

(defun tautologyp-helper (formula assignments)
  (if (endp assignments)
      t
    (if (eval-formula formula (car assignments))
        (tautologyp-helper formula (cdr assignments))
      nil)))

In [ ]:
; Tautology: p OR NOT p → ((1) (-1)) is NOT a tautology in CNF!
; In CNF, ((1) (-1)) means (x1) AND (NOT x1) which is UNSAT.
; The tautology p OR NOT p is a single clause: ((1 -1))
(tautologyp '((1 -1)) 1)

In [ ]:
; ((1 -1)) is indeed a tautology (the clause x1 OR NOT x1)
; ((1) (-1)) is unsatisfiable (x1 AND NOT x1)
(list (tautologyp '((1 -1)) 1)      ; t (tautology)
      (tautologyp '((1) (-1)) 1))    ; nil (not a tautology)

The truth-table approach works but is exponential in the number of variables ($2^n$ assignments). For practical formulas with thousands of variables, we need smarter algorithms.

## 8.4 Unit Propagation

Unit propagation is a core technique in modern SAT solvers. The idea is simple:

> If a clause has only **one unassigned literal**, that literal **must be true** for the clause to be satisfied.

This forced assignment may simplify other clauses, triggering further propagations.

In [ ]:
; Check if a clause is a unit clause under an assignment.
; A unit clause has exactly one unassigned literal and
; all other literals evaluate to false.
(defun find-unit-literal (clause assignment)
  (find-unit-literal-helper clause assignment nil))

(defun find-unit-literal-helper (clause assignment unassigned-lit)
  (if (endp clause)
      unassigned-lit  ; returns nil if 0 unassigned, the literal if exactly 1
    (let ((lit (car clause)))
      (if (eval-literal lit assignment)
          nil  ; clause already satisfied
        (let ((var (if (< lit 0) (- lit) lit)))
          (if (lookup-var var assignment)
              ;; variable is assigned but literal is false, skip
              (find-unit-literal-helper (cdr clause) assignment unassigned-lit)
            ;; variable is unassigned
            (if unassigned-lit
                nil  ; more than one unassigned → not unit
              (find-unit-literal-helper (cdr clause) assignment lit))))))))

### Performing Unit Propagation

When we find a unit literal, we add the forced assignment and repeat:

In [ ]:
; Assign a literal: if lit > 0, set var to t; if lit < 0, set var to nil
(defun assign-literal (lit assignment)
  (if (< lit 0)
      (cons (cons (- lit) nil) assignment)
    (cons (cons lit t) assignment)))

In [ ]:
; Find any unit clause in a formula and return its forced literal,
; or nil if no unit clause exists.
(defun find-unit-in-formula (formula assignment)
  (if (endp formula)
      nil
    (let ((unit-lit (find-unit-literal (car formula) assignment)))
      (if unit-lit
          unit-lit
        (find-unit-in-formula (cdr formula) assignment)))))

In [ ]:
; Perform unit propagation: repeatedly find and assign unit literals.
; Returns the extended assignment.
; Uses a step counter to ensure termination.
(defun unit-propagate (formula assignment steps)
  (if (zp steps)
      assignment
    (let ((unit-lit (find-unit-in-formula formula assignment)))
      (if unit-lit
          (unit-propagate formula
                         (assign-literal unit-lit assignment)
                         (- steps 1))
        assignment))))

### Soundness of Unit Propagation

The key property of unit propagation is that it **preserves satisfiability**: if the original formula is satisfiable, it remains satisfiable after forced assignments.

Formally, any satisfying assignment must agree with the forced assignments (otherwise a unit clause would be violated).

In [ ]:
; Test unit propagation on ((1) (-1 2) (-2 3))
; Clause (1) forces x1=t.
; Then (-1 2) becomes a unit: since x1=t, -1 is false, so 2 is forced → x2=t.
; Then (-2 3) becomes a unit: since x2=t, -2 is false, so 3 is forced → x3=t.
(unit-propagate '((1) (-1 2) (-2 3)) nil 10)

## 8.5 A Simple DPLL Solver

The **Davis–Putnam–Logemann–Loveland (DPLL)** algorithm is the foundation of all modern SAT solvers. It combines:

1. **Unit propagation** to simplify the formula
2. **Branching** — pick an unassigned variable, try both true and false
3. **Backtracking** — if one branch fails, try the other

DPLL returns either a satisfying assignment or `'UNSAT`.

In [ ]:
; Check if a formula is satisfied by an assignment
(defun formula-satisfied-p (formula assignment)
  (eval-formula formula assignment))

In [ ]:
; Check if any clause is falsified (all its literals are false
; under the current assignment, with all variables assigned)
(defun clause-falsified-p (clause assignment)
  (if (endp clause)
      t  ; empty clause is falsified
    (if (eval-literal (car clause) assignment)
        nil  ; found a true literal
      (let ((var (if (< (car clause) 0) (- (car clause)) (car clause))))
        (if (lookup-var var assignment)
            (clause-falsified-p (cdr clause) assignment)
          nil)))))  ; unassigned variable → not yet falsified

In [ ]:
; Check if any clause in the formula is falsified
(defun has-falsified-clause (formula assignment)
  (if (endp formula)
      nil
    (if (clause-falsified-p (car formula) assignment)
        t
      (has-falsified-clause (cdr formula) assignment))))

In [ ]:
; Pick the first unassigned variable from a formula.
; Scans all literals to find a variable not in the assignment.
(defun pick-variable (formula assignment)
  (pick-variable-from-clauses formula assignment))

(defun pick-variable-from-clauses (formula assignment)
  (if (endp formula)
      nil
    (let ((v (pick-variable-from-clause (car formula) assignment)))
      (if v v
        (pick-variable-from-clauses (cdr formula) assignment)))))

(defun pick-variable-from-clause (clause assignment)
  (if (endp clause)
      nil
    (let* ((lit (car clause))
           (var (if (< lit 0) (- lit) lit)))
      (if (lookup-var var assignment)
          (pick-variable-from-clause (cdr clause) assignment)
        var))))

In [ ]:
; The DPLL SAT solver.
; Uses a step counter for termination.
(defun dpll (formula assignment steps)
  (if (zp steps)
      'UNKNOWN
    (let ((assignment (unit-propagate formula assignment 100)))
      (cond
       ;; Check if formula is satisfied
       ((formula-satisfied-p formula assignment)
        assignment)
       ;; Check if any clause is falsified → backtrack
       ((has-falsified-clause formula assignment)
        'UNSAT)
       ;; Branch on an unassigned variable
       (t
        (let ((var (pick-variable formula assignment)))
          (if (not var)
              ;; No unassigned variables left
              (if (formula-satisfied-p formula assignment)
                  assignment
                'UNSAT)
            ;; Try var = t first
            (let ((result (dpll formula
                               (cons (cons var t) assignment)
                               (- steps 1))))
              (if (not (equal result 'UNSAT))
                  result
                ;; Try var = nil
                (dpll formula
                      (cons (cons var nil) assignment)
                      (- steps 1)))))))))))

### Testing the DPLL Solver

In [ ]:
; Satisfiable: (x1 OR x2) AND (NOT x1 OR x3) AND (NOT x3 OR x2)
(dpll '((1 2) (-1 3) (-3 2)) nil 100)

In [ ]:
; Unsatisfiable: (x1) AND (NOT x1)
(dpll '((1) (-1)) nil 100)

In [ ]:
; A more complex satisfiable formula:
; (x1 OR x2) AND (NOT x1 OR NOT x2) AND (x1 OR NOT x2) 
; Satisfiable with x1=t, x2=nil
(dpll '((1 2) (-1 -2) (1 -2)) nil 100)

### Soundness of DPLL

The key soundness property is: **if DPLL returns an assignment, that assignment satisfies the formula.**

In [ ]:
; Verify the answer: check that the returned assignment
; actually satisfies the formula.
(let ((result (dpll '((1 2) (-1 3) (-3 2)) nil 100)))
  (if (not (equal result 'UNSAT))
      (eval-formula '((1 2) (-1 3) (-3 2)) result)
    'was-unsat))

## 8.6 SAT in the ACL2 Community Books

The ACL2 community has developed industrial-strength SAT tools:

### Verified SAT Proof Checkers

Modern SAT solvers can emit **proof certificates** (LRAT/DRAT format) that attest to unsatisfiability. ACL2's verified checkers can validate these certificates with mathematical certainty:

- **`books/projects/sat/lrat/`** — A verified LRAT (Linear Resolution Asymmetric Tautology) proof checker
- The checker itself is proved correct in ACL2: if it accepts a proof, the formula is truly unsatisfiable
- This gives a **verified pipeline**: untrusted SAT solver → proof → verified checker

### SATLink

**`books/centaur/satlink/`** provides an interface between ACL2 and external SAT solvers (like Glucose, CryptoMiniSat, etc.). It:

1. Translates ACL2 boolean formulas to DIMACS CNF format
2. Calls an external solver
3. Validates the result using a verified proof checker

### SULFA

**`books/clause-processors/SULFA/`** provides a Subclass of Unrollable List Formulas in ACL2 — a way to reduce ACL2 conjectures to SAT problems.

## 8.7 BDDs (Binary Decision Diagrams)

A **Binary Decision Diagram (BDD)** is a compact data structure for representing boolean functions. Unlike CNF, BDDs support efficient equivalence checking — two BDDs represent the same function if and only if they are structurally equal.

ACL2 has built-in support for BDD-based reasoning through its `IF`-then-else normal form.

### How BDDs Work

A BDD is a directed acyclic graph where:
- Each internal node is a variable that branches on true/false
- Leaf nodes are `T` or `NIL`
- Variables appear in a fixed order on all paths
- Isomorphic sub-graphs are shared (canonicity)

For a function $f(x_1, x_2, x_3)$, the BDD encodes the **Shannon expansion**:

$$f = (x_1 \wedge f|_{x_1=1}) \vee (\neg x_1 \wedge f|_{x_1=0})$$

In [ ]:
; ACL2 can reason about IF-then-else trees directly.
; This is related to BDD reasoning.
; Example: verify that two IF-expressions compute the same function.
(thm
  (equal (if a (if b t nil) nil)
         (if b (if a t nil) nil)))

In [ ]:
; A more complex equivalence:
; (a AND b) OR (a AND NOT b) = a
(thm
  (equal (if (if a (if b t nil) nil)
             t
           (if (if a (if b nil t) nil)
               t
             nil))
         (if a t nil)))

### BDDs in the Community Books

The **`books/centaur/ubdds/`** library provides a uniquified BDD package for ACL2 with:

- Efficient BDD construction and manipulation
- Verified canonicity properties
- Integration with hardware verification tools

BDDs complement SAT solving: BDDs work well for equivalence checking of moderate-size circuits, while SAT excels at checking constraints on large circuits.

## 8.8 Exercises

### Exercise 8.1: Formula Construction

Encode the following formula in CNF and check it with `eval-formula`:

$$(x_1 \Rightarrow x_2) \wedge (x_2 \Rightarrow x_3) \wedge x_1 \wedge \neg x_3$$

Recall that $A \Rightarrow B$ is equivalent to $\neg A \vee B$. Is this formula satisfiable?

In [ ]:
; Exercise 8.1: Encode and check the formula
; (x1 => x2) AND (x2 => x3) AND x1 AND NOT x3
;
; Hint: x1 => x2 becomes clause (-1 2)
;
; YOUR CODE HERE
; (dpll '(...) nil 100)

### Exercise 8.2: Resolution

The **resolution rule** is: from clauses $(A \vee x)$ and $(B \vee \neg x)$, derive $(A \vee B)$.

Define a function `resolve` that takes two clauses and a variable, and returns the resolvent. Then verify that if both input clauses are satisfied, the resolvent is also satisfied.

In [ ]:
; Exercise 8.2: Resolution
;
; (defun resolve (clause1 clause2 var)
;   ;; Remove +var from clause1, -var from clause2,
;   ;; return the union of remaining literals
;   YOUR DEFINITION)
;
; Test: resolve '(1 2) '(-1 3) 1 → (2 3)

### Exercise 8.3: Pure Literal Elimination

A literal is **pure** in a formula if it appears only positively (or only negatively). Pure literals can be assigned to make their clauses true.

Define `find-pure-literal` that scans a formula and returns a pure literal if one exists.

In [ ]:
; Exercise 8.3: Pure literal elimination
;
; (defun find-pure-literal (formula)
;   ;; Scan all literals. If a variable appears only positive
;   ;; or only negative, return that literal.
;   YOUR DEFINITION)
;
; Test: in ((1 2) (1 3) (-2 3)), literal 1 is pure (always positive)
; (find-pure-literal '((1 2) (1 3) (-2 3)))  → 1 or 3

### Exercise 8.4: Counting Solutions

Define `count-solutions` that counts how many assignments (out of all $2^n$) satisfy a given formula. Use the `all-assignments` function.

In [ ]:
; Exercise 8.4: Count solutions
;
; (defun count-solutions (formula n)
;   ;; Count how many of the 2^n assignments satisfy formula
;   YOUR DEFINITION)
;
; Test: (1 -1) is a tautology over 1 variable → 2 solutions
; (count-solutions '((1 -1)) 1) → 2
;
; (1) AND (-1) over 1 variable → 0 solutions
; (count-solutions '((1) (-1)) 1) → 0

---

**Navigation:**
[< Module 7 — Hardware Verification](07_hardware_verification.ipynb) | [Module 9 — Cryptography and Security >](09_crypto_security.ipynb)